[주제]
일부 레이블만 주어진 학습 데이터셋을 이용한 항공편 지연 여부 예측



[설명]
레이블이 없는 데이터와 함께 항공편 지연 여부를 예측하는 AI 모델

모델 : catboost

구성:
1. CSV -> Parquet 변환 (1회 실행용)
2. Parquet 로딩 (빠르고 메모리 효율적)
3. Labeled 데이터만 사용하여 CatBoost 학습
4. 결측 플래그 추가 (missing indicator)
5. 검증(LogLoss) + 제출 파일 생성

패키지 임포트

In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

! pip install catboost

from catboost import CatBoostClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:00


## Data Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
file_dir = '/content/drive/MyDrive/월간 데이콘 항공편 지연 예측 AI 경진대회 데이터'

재생산성(시드 고정)

In [ ]:
# 시드 고정 = 난수 기반의 실험 결과의 일관성을 유지하고 반복 가능. Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

Parquet를 만드는 이유(바이너리)
- 데이터가 너무 큼
- 로딩이 빠름
- 전처리 반복 비용을 줄여줌
- 필요한 컬럼만 빨리 로딩할 수 있음

In [ ]:
train_csv_path = file_dir + "/train.csv"
test_csv_path  = file_dir + "/test.csv"
sub_csv_path   = file_dir + "/sample_submission.csv"

In [ ]:
train_pq_path = "./train.parquet"
test_pq_path  = "./test.parquet"

In [ ]:
def csv_to_parquet(csv_path, save_name):
    """
    csv_path의 CSV를 읽어서 ./{save_name}.parquet로 저장합니다.
    save_name 예: 'train' -> './train.parquet' 생성
    """
    df = pd.read_csv(csv_path)
    df.to_parquet(f'./{save_name}.parquet', index=False)  # index=False 추천(불필요한 인덱스 저장 방지)
    del df
    gc.collect()
    print(save_name, 'Done.')

In [ ]:
csv_to_parquet(train_csv_path, 'train')
csv_to_parquet(test_csv_path, 'test')
sub_csv_path = file_dir + "/sample_submission.csv"
sample_submission = pd.read_csv(sub_csv_path, index_col = 0)

train Done.
test Done.


In [ ]:
train = pd.read_parquet(train_pq_path)
test = pd.read_parquet(test_pq_path)

## Data Processing

In [ ]:
train.head(10)

,ID,Month,Day_of_Month,Estimated_Departure_Time,Estimated_Arrival_Time,Cancelled,Diverted,Origin_Airport,Origin_Airport_ID,Origin_State,Destination_Airport,Destination_Airport_ID,Destination_State,Distance,Airline,Carrier_Code(IATA),Carrier_ID(DOT),Tail_Number,Delay
0,TRAIN_000000,4,15,NaN,NaN,0,0,OKC,13851,Oklahoma,HOU,12191,Texas,419.0,Southwest Airlines Co.,WN,19393.0,N7858A,None
1,TRAIN_000001,8,15,740.0,1024.0,0,0,ORD,13930,Illinois,SLC,14869,Utah,1250.0,SkyWest Airlines Inc.,UA,20304.0,N125SY,None
2,TRAIN_000002,9,6,1610.0,1805.0,0,0,CLT,11057,North Carolina,LGA,12953,New York,544.0,American Airlines Inc.,AA,19805.0,N103US,None
3,TRAIN_000003,7,10,905.0,1735.0,0,0,LAX,12892,California,EWR,11618,New Jersey,2454.0,United Air Lines Inc.,UA,NaN,N595UA,None
4,TRAIN_000004,1,11,900.0,1019.0,0,0,SFO,14771,California,ACV,10157,California,250.0,SkyWest Airlines Inc.,UA,20304.0,N161SY,None
5,TRAIN_000005,4,13,1545.0,NaN,0,0,EWR,11618,None,DCA,11278,Virginia,199.0,Republic Airlines,UA,20452.0,N657RW,Not_Delayed
6,TRAIN_000006,1,20,1742.0,1903.0,0,0,EWR,11618,New Jersey,BOS,10721,Massachusetts,200.0,United Air Lines Inc.,UA,NaN,N66825,Not_Delayed
7,TRAIN_000007,4,20,1815.0,1955.0,0,0,ORD,13930,Illinois,MCI,13198,Missouri,403.0,None,UA,20304.0,N110SY,None
8,TRAIN_000008,6,13,1420.0,1550.0,0,0,BWI,10821,None,CLT,11057,North Carolina,361.0,Southwest Airlines Co.,WN,19393.0,N765SW,Not_Delayed
9,TRAIN_000009,6,6,650.0,838.0,0,0,LIT,12992,Arkansas,IAH,12266,Texas,374.0,ExpressJet Airlines Inc.,UA,20366.0,N14902,None


In [ ]:
train.info()
print("train:", train.shape, "test:", test.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 19 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   ID                        1000000 non-null  object 
 1   Month                     1000000 non-null  int64  
 2   Day_of_Month              1000000 non-null  int64  
 3   Estimated_Departure_Time  890981 non-null   float64
 4   Estimated_Arrival_Time    890960 non-null   float64
 5   Cancelled                 1000000 non-null  int64  
 6   Diverted                  1000000 non-null  int64  
 7   Origin_Airport            1000000 non-null  object 
 8   Origin_Airport_ID         1000000 non-null  int64  
 9   Origin_State              890985 non-null   object 
 10  Destination_Airport       1000000 non-null  object 
 11  Destination_Airport_ID    1000000 non-null  int64  
 12  Destination_State         890921 non-null   object 
 13  Distance                  10

In [ ]:
# 결측치 알아보기
(train.isna().mean().sort_values(ascending=False).head(10))
# miss_values = train.isnull().sum()
# miss_values

,0
Delay,0.744999
Destination_State,0.109079
Estimated_Arrival_Time,0.109040
Estimated_Departure_Time,0.109019
Origin_State,0.109015
Carrier_ID(DOT),0.108997
Carrier_Code(IATA),0.108990
Airline,0.108920
Day_of_Month,0.000000
ID,0.000000


In [ ]:
train_miss_values = train.isnull().sum()
train_miss_values

,0
ID,0
Month,0
Day_of_Month,0
Estimated_Departure_Time,109019
Estimated_Arrival_Time,109040
Cancelled,0
Diverted,0
Origin_Airport,0
Origin_Airport_ID,0
Origin_State,109015


In [ ]:
test_miss_values = test.isnull().sum()
test_miss_values

,0
ID,0
Month,0
Day_of_Month,0
Estimated_Departure_Time,108984
Estimated_Arrival_Time,109048
Cancelled,0
Diverted,0
Origin_Airport,0
Origin_Airport_ID,0
Origin_State,106505


In [ ]:
# 컬럼 지정
TARGET_COL = "Delay"  # 타겟 컬럼
ID_COL = "ID"         # ID 컬럼

# sample_submission에서 예측 확률을 넣을 컬럼 이름
# index_col=0을 사용했으므로, 예측 확률을 넣을 컬럼은 보통 columns[0]
PROB_COL = sample_submission.columns[0]
print("[INFO] PROB_COL:", PROB_COL)

[INFO] PROB_COL: Not_Delayed


In [ ]:
# labeled rows only
# Delay가 NaN이면 정답이 없어서 학습에 직접 사용 불가

labeled = train[train[TARGET_COL].notna()].copy()

print("[INFO] labeled rows:", len(labeled))
print("[INFO] unlabeled rows:", train[TARGET_COL].isna().sum())

# labeled에서 정답을 빼고 남긴게 x = 모델에 입력으로 넣을 feature 모음
# ML = 입력 / 정답 분리가 필요. 입력 : x(항공사, 시간, 주 공항) / 정답 : y (지연 여부)
# Delay를 x에 남겨두면 모델이 정답을 그대로 보고 학습해서 데이터 누수가 발생
X = labeled.drop(columns=[TARGET_COL])

# "Not_Delayed", "Delayed" -> 현재 문자열. 숫자로 바꾸는 라벨 인코딩이 필요
# 오류 발생 : astype(int) 하려는데 값이 문자열이라서 int로 바꾸지 못해 에러 발생
# 해결 : 숫자로 인코딩하여 int 로 변환. 숫자 형태여야 안정적으로 학습 가능하다는 것을 께달음

y = labeled[TARGET_COL].map({"Not_Delayed": 0, "Delayed": 1})

# 안전장치: 혹시 다른 라벨이 있으면 즉시 알려주고 멈춤
if y.isna().any():
    unknown = labeled.loc[y.isna(), TARGET_COL].unique()
    print("[ERROR] Unknown Delay labels:", unknown)
    raise ValueError("Delay 라벨 매핑이 누락되었습니다. map 딕셔너리를 수정하세요.")

y = y.astype(int)

X_test = test.copy()

[INFO] labeled rows: 255001
[INFO] unlabeled rows: 744999


In [ ]:
if ID_COL in X.columns:
    X = X.drop(columns=[ID_COL])
if ID_COL in X_test.columns:
    X_test = X_test.drop(columns=[ID_COL])

In [ ]:
# 결측치가 존재하는 feature에 대해 isnull 컬럼을 추가

cols_with_missing = [c for c in X.columns if X[c].isna().any()]
print("[INFO] columns with missing:", cols_with_missing)

for c in cols_with_missing:
    X[c + "_isnull"] = X[c].isna().astype(int)
    X_test[c + "_isnull"] = X_test[c].isna().astype(int)

[INFO] columns with missing: ['Estimated_Departure_Time', 'Estimated_Arrival_Time', 'Origin_State', 'Destination_State', 'Airline', 'Carrier_Code(IATA)', 'Carrier_ID(DOT)']


In [ ]:
# 3/3 시간 파생변수 추가
def _parse_hhmm_to_min(x):

    # 'HH:MM' 또는 'H:MM' 형태를 분(min) 단위로 변환.

    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    try:
        hh, mm = s.split(":")
        hh = int(hh)
        mm = int(mm)
        if not (0 <= hh <= 23 and 0 <= mm <= 59):
            return np.nan
        return hh * 60 + mm
    except:
        return np.nan

In [ ]:
def _time_bin_from_hour(h):
    if pd.isna(h):
        return "missing"
    h = int(h)
    if h <= 5:
        return "night"      # 0~5
    if h <= 11:
        return "morning"    # 6~11
    if h <= 17:
        return "afternoon"  # 12~17
    return "evening"        # 18~23

In [ ]:
def add_time_features(df):
    # df에 아래 파생변수 추가:
    # - Estimated_Departure_Time_min, _hour, _minute, _bin
    # - Estimated_Arrival_Time_min,   _hour, _minute, _bin
    # - ETA_minus_ETD_min: (도착 - 출발) 분 단위. 자정 넘어가는 경우 보정(+1440)
    dep_col = "Estimated_Departure_Time"
    arr_col = "Estimated_Arrival_Time"

    # 1) 분 단위 변환 (0~1439)
    if dep_col in df.columns:
        df[dep_col + "_min"] = df[dep_col].apply(_parse_hhmm_to_min)
        df[dep_col + "_hour"] = (df[dep_col + "_min"] // 60)
        df[dep_col + "_minute"] = (df[dep_col + "_min"] % 60)
        df[dep_col + "_bin"] = df[dep_col + "_hour"].apply(_time_bin_from_hour).astype(str)

    if arr_col in df.columns:
        df[arr_col + "_min"] = df[arr_col].apply(_parse_hhmm_to_min)
        df[arr_col + "_hour"] = (df[arr_col + "_min"] // 60)
        df[arr_col + "_minute"] = (df[arr_col + "_min"] % 60)
        df[arr_col + "_bin"] = df[arr_col + "_hour"].apply(_time_bin_from_hour).astype(str)

    # 2) (도착 - 출발) 추정 소요시간(분) 파생
    #    - 자정 넘어가는 항공편(예: 23:50 -> 01:10)은 음수가 나오므로 +1440 보정
    if (dep_col + "_min") in df.columns and (arr_col + "_min") in df.columns:
        diff = df[arr_col + "_min"] - df[dep_col + "_min"]
        diff = np.where(diff < 0, diff + 1440, diff)  # 자정 넘어가면 하루(1440분) 더함
        df["ETA_minus_ETD_min"] = diff
        # 결측이 있으면 diff도 NaN일 수 있으니 그대로 둠
        df["ETA_minus_ETD_min"] = pd.to_numeric(df["ETA_minus_ETD_min"], errors="coerce")

    return df

In [ ]:
X = add_time_features(X)
X_test = add_time_features(X_test)

In [ ]:
print(train["Estimated_Departure_Time"].dropna().head())
print(train["Estimated_Arrival_Time"].dropna().head())

1     740.0
2    1610.0
3     905.0
4     900.0
5    1545.0
Name: Estimated_Departure_Time, dtype: float64
1    1024.0
2    1805.0
3    1735.0
4    1019.0
6    1903.0
Name: Estimated_Arrival_Time, dtype: float64


In [ ]:
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

maybe_cat = [
    "Airline",
    "Carrier_Code(IATA)",
    "Carrier_ID(DOT)",
    "Origin_State",
    "Destination_State",
]
for c in maybe_cat:
    if c in X.columns and c not in cat_cols:
        cat_cols.append(c)

# cat_features는 컬럼 인덱스로 전달
cat_features = [X.columns.get_loc(c) for c in cat_cols]

print("[INFO] #categorical cols:", len(cat_cols))
print("[INFO] categorical cols (sample):", cat_cols[:10])


[INFO] #categorical cols: 10
[INFO] categorical cols (sample): ['Origin_Airport', 'Origin_State', 'Destination_Airport', 'Destination_State', 'Airline', 'Carrier_Code(IATA)', 'Tail_Number', 'Estimated_Departure_Time_bin', 'Estimated_Arrival_Time_bin', 'Carrier_ID(DOT)']


In [ ]:
for c in cat_cols:
    X[c] = X[c].astype("object").fillna("missing")
    X_test[c] = X_test[c].astype("object").fillna("missing")


In [ ]:
# 학습
# model.fit(
#     X_tr, y_tr,
#     cat_features=cat_features,
#     eval_set=(X_va, y_va),
#     use_best_model=True,
#     early_stopping_rounds=200
# )

# 오류 : 내가 지정한 cat_features 안에 값이 실수여서 catboost는 float를 허용하지 않는다고 해서 오류발생.
# catboost는 문자열 또는 정수를 허용함.
# 숫자 컬럼에 결측치가 섞이면 pandas 가 종종 dtype을 float로 설정한다고 함.

# cat_features로 지정한 컬럼은 전부 문자열로 바꾸기
for c in cat_cols:
    # NaN 먼저 missing으로
    X[c] = X[c].fillna("missing")
    X_test[c] = X_test[c].fillna("missing")

    # 전부 문자열로 강제 변환 (float -> "20253.0"도 문자열이 되므로 안전)
    X[c] = X[c].astype(str)
    X_test[c] = X_test[c].astype(str)

for c in cat_cols:
  X_tr[c] = X_tr[c].fillna("missing").astype(str)
  X_va[c] = X_va[c].fillna("missing").astype(str)

In [ ]:
# evaluation (LogLoss)
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)
print("[INFO] train split:", X_tr.shape, "valid split:", X_va.shape)


[INFO] train split: (204000, 33) valid split: (51001, 33)


In [ ]:
# catboost model
# model = CatBoostClassifier(
#     loss_function="Logloss",
#     eval_metric="Logloss",
#     iterations=5000,
#     learning_rate=0.03,
#     depth=8,
#     l2_leaf_reg=3.0,
#     random_seed=SEED,
#     verbose=200,
#     auto_class_weights="Balanced",
# )

In [ ]:
# model.fit(
#     X_tr, y_tr,
#     cat_features=cat_features,
#     eval_set=(X_va, y_va),
#     use_best_model=True,
#     early_stopping_rounds=200
# )

In [ ]:
# model.fit(
#     X_tr, y_tr,
#     cat_features=cat_features,
#     eval_set=(X_va, y_va),
#     use_best_model=True,
#     early_stopping_rounds=200
# )
# 결과 : bestTest = 0.6452250966
# bestIteration = 1111

# -> 과적합 방지. 검증 성능이 더이상 개선되지 않아서 멈춰버림.
# 학습이 진행된건 맞는데 1111 트리 이후에 검증 성능이 좋아질 게 없어서 멈춤

# 전처리/타겟 확인 방법
# 1. 타겟 클래스가 0/1로만 구성되어있는지
# print("y unique:", np.sort(y.unique()))
# print("y mean (positive rate):", y.mean())

# # 2. train과 test 컬럼이 동일한지
# print("X cols == X_test cols:", set(X.columns) == set(X_test.columns)) # true. 동일

# va_pred = model.predict_proba(X_va)[:, 1]
# print("pred min/max/mean:", va_pred.min(), va_pred.max(), va_pred.mean())

y unique: [0 1]
y mean (positive rate): 0.1764698961964855
X cols == X_test cols: True


NameError: name 'model' is not defined

In [ ]:
model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="Logloss",
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3.0,
    random_seed=SEED,
    verbose=200,
    auto_class_weights="None",
)

In [ ]:
model.fit(
    X_tr, y_tr,
    cat_features=cat_features,
    eval_set=(X_va, y_va),
    use_best_model=True,
    early_stopping_rounds=200
)

0:	learn: 0.6777719	test: 0.6777859	best: 0.6777859 (0)	total: 508ms	remaining: 42m 18s
200:	learn: 0.4399644	test: 0.4425785	best: 0.4425785 (200)	total: 1m 50s	remaining: 43m 52s
400:	learn: 0.4336767	test: 0.4401955	best: 0.4401955 (400)	total: 3m 47s	remaining: 43m 25s
600:	learn: 0.4273089	test: 0.4384782	best: 0.4384777 (598)	total: 5m 45s	remaining: 42m 6s
800:	learn: 0.4225313	test: 0.4378821	best: 0.4378805 (793)	total: 7m 40s	remaining: 40m 14s
1000:	learn: 0.4180348	test: 0.4375442	best: 0.4375433 (999)	total: 9m 40s	remaining: 38m 37s
1200:	learn: 0.4140791	test: 0.4374207	best: 0.4374207 (1200)	total: 11m 37s	remaining: 36m 46s
1400:	learn: 0.4104173	test: 0.4373330	best: 0.4373020 (1373)	total: 13m 35s	remaining: 34m 55s
1600:	learn: 0.4067818	test: 0.4372069	best: 0.4372069 (1600)	total: 15m 36s	remaining: 33m 7s
1800:	learn: 0.4031581	test: 0.4371857	best: 0.4371852 (1789)	total: 17m 34s	remaining: 31m 12s
2000:	learn: 0.3995545	test: 0.4372313	best: 0.4371850 (1805)	to

CatBoostClassifier(auto_class_weights='None', depth=8, eval_metric='Logloss', iterations=5000, l2_leaf_reg=3.0, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=200)

In [ ]:
# feature에 target이 섞였는지 최종 확인
print("TARGET in X columns?", "Delay" in X.columns)

# train/test 컬럼 동일 확인
print(set(X.columns) == set(X_test.columns))

# 예측 확률 분포 확인
va_pred = model.predict_proba(X_va)[:,1]
print("pred min/max/mean:", va_pred.min(), va_pred.max(), va_pred.mean())

TARGET in X columns? False
True
pred min/max/mean: 0.009790555984281258 0.7522892911687779 0.17431530369195905
